# Parche de módulo

Agrega o corrige **un** módulo sin rehacer la actualización anual (unos minutos por módulo):

1. Crea `parches/<id_modulo>/` copiando `parches/_plantilla/` y edita `modulo.yml` y `diagnosticos.csv`
   (opcional: `procedimientos.csv`, `controles.csv` con las cifras del artículo). Ver `parches/LEEME.md`.
2. Escribe el id abajo y ejecuta todo (**Run All**).
3. Revisa los controles y la app. Cuando estén bien, cambia `estado: disponible` en `modulo.yml` y vuelve a correr
   (o solo la celda 1 y la 4, porque los datos no cambian).
4. Sube el zip en el panel (Publicar, Actualización anual) o la carpeta para GitHub de la celda 5.

Requiere haber corrido antes `actualizacion_anual.ipynb` (usa su `config_local.json` con las rutas DEIS y GRD).

In [ ]:
import sys, json
from pathlib import Path
# El repositorio se detecta solo si abres el notebook desde su carpeta notebooks/.
REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "pipeline" / "cmf_local.py").exists() else Path(r"C:/Users/gusta/Downloads/observatoriocmf/observatoriocmf")
sys.path.insert(0, str(REPO_DIR / "pipeline"))
import importlib, cmf_local as cmf; importlib.reload(cmf)
TRABAJO = REPO_DIR.parent / "observatorio_corrida_anual"   # copia de trabajo de la actualización anual
RSCRIPT = None   # ruta a Rscript.exe si no está en el PATH
print("Repositorio:", REPO_DIR, "| copia de trabajo:", TRABAJO, TRABAJO.exists())

MODULO = "cancer_oral"          # carpeta parches/<MODULO>/
PARCHE = REPO_DIR / "parches" / MODULO
assert (PARCHE / "modulo.yml").exists(), f"Falta {PARCHE / 'modulo.yml'} (copia parches/_plantilla)"
assert (TRABAJO / "config_local.json").exists(), "Falta config_local.json: corre primero actualizacion_anual.ipynb"

## 1. Aplicar la definición (repositorio y copia de trabajo)

In [ ]:
import definicion_modulo as dm; importlib.reload(dm)
dm.aplicar(PARCHE, REPO_DIR)
cmf.sincronizar_codigo(REPO_DIR, TRABAJO)
# Validación: el YAML debe leerse y el módulo debe tener códigos incluidos.
cmf.correr_r(["-e", f"m <- yaml::read_yaml('config/modules.yml')$modulos; ids <- vapply(m, `[[`, '', 'id'); "
                    f"stopifnot('{MODULO}' %in% ids); cat('modules.yml OK:', length(ids), 'módulos\\n')"], TRABAJO, r=RSCRIPT)

## 2. Reclasificar el módulo y recalcular las tablas (fase 2)

In [ ]:
import datetime
t0 = datetime.datetime.now()
cmf.correr_r(["pipeline/parche_modulo.R", "config_local.json", MODULO], TRABAJO, log=f"log_parche_{MODULO}.txt", r=RSCRIPT, mostrar=25)
print("Duración:", datetime.datetime.now() - t0)

## 3. Revisar: controles y egresos por año

In [ ]:
import pandas as pd
ext = TRABAJO / "inst" / "extdata"
ct = pd.DataFrame(json.loads((ext / "controles.json").read_text(encoding="utf-8")))
pd.set_option("display.max_colwidth", 90)
display(ct[ct.modulo.eq(MODULO)][["referencia", "anio_desde", "anio_hasta", "valor", "obtenido", "estado"]] if "modulo" in ct else ct)
cubo = pd.read_csv(ext / "cubo.csv.gz", usecols=["fuente", "modulo", "subgrupo", "anio", "n"], low_memory=False)
c = cubo[cubo.modulo.eq(MODULO)]
display(c.pivot_table(index="anio", columns="fuente", values="n", aggfunc="sum").fillna(0).astype(int))
display(c.groupby(["fuente", "subgrupo"]).n.sum().unstack(0).fillna(0).astype(int))

## 4. Ver la app, copiar al repositorio y empaquetar

In [ ]:
app = cmf.abrir_app(TRABAJO, r=RSCRIPT)      # app.terminate() para cerrarla

In [ ]:
cmf.publicar_en_repo(TRABAJO, REPO_DIR)      # tablas publicadas y casos clasificados al repositorio
zip_path = cmf.zip_panel(TRABAJO)            # para el panel: Publicar, Actualización anual

## 5. Carpeta para subir a GitHub

Carpeta limpia junto al repositorio (`observatoriocmf_github`): arrástrala en github.com o ábrela con GitHub Desktop. `CON_CASOS = True` agrega `data-casos/` (necesario para *Recalcular* en GitHub; son registros por egreso sin identificadores ni fechas).

In [ ]:
CON_CASOS = False
cmf.carpeta_github(REPO_DIR, con_casos=CON_CASOS)